# 01 · Evaluation setup + data store  (steps 1-3, CPU session)
**Gate (step 1):** the released PEdger checkpoint must reproduce its published ODS within ±0.002
with our Python benchmark before any number from this pipeline is trusted.
The released checkpoint was saved at epoch 19 and its training set is not stated; compare against
the published BSDS+VOC (.821/.840) and BSDS-only (.813/.834) numbers and note which one it matches.
Output of this notebook (`/kaggle/working/data_store`) → save as Kaggle Dataset **pedger-data**.

In [ ]:
# --- setup: code + upstream PEdger -------------------------------------------
import os, subprocess, sys
REPO_URL = ""          # e.g. "https://github.com/<you>/pedger-mtkd.git" (or attach the code as a Kaggle Dataset)
CODE_DATASET = "/kaggle/input/pedger-mtkd-code"
W = "/kaggle/working"
REPO = f"{W}/pedger-mtkd"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run(["cp", "-r", CODE_DATASET, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py
!pip install -q -r requirements.txt

In [ ]:
# BSDS500 original release (per-annotator .mat ground truth)
RAW_BSDS = "/kaggle/working/BSR/BSDS500/data"
if not os.path.exists(RAW_BSDS):
    !wget -q https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz -O /kaggle/working/bsr.tgz
    !tar -xzf /kaggle/working/bsr.tgz -C /kaggle/working && rm /kaggle/working/bsr.tgz
!ls {RAW_BSDS}/images

## Steps 2-3: array store + agreement maps (annotator fraction computed from the per-annotator .mat files)

In [ ]:
!python data/prepare_bsds.py --bsds-root {RAW_BSDS} --out /kaggle/working/data_store/bsds
!cat /kaggle/working/data_store/bsds/meta.json

In [ ]:
# Visual check (step 3 gate: maps look right on 10 images): image | a | Y | alpha
!python data/visualize_samples.py --store /kaggle/working/data_store/bsds --split train --n 10 --out /kaggle/working/checks/agreement.png
from IPython.display import Image; Image("/kaggle/working/checks/agreement.png")

## Step 1: score the released PEdger checkpoint in every inference mode
`ns_m` = non-recurrent momentum net (0.73M, the deployed model), `rec_m` = recurrent, `fuse_m` = both fused.
`--ms` = multi-scale. Pick the mode that reproduces the paper and set `eval.deploy` in `configs/base.yaml`.

In [ ]:
CKPT = "third_party/PEdger/checkpoint_PEdger.pth"
for mode in ["ns_m", "rec_m", "fuse_m"]:
    for ms in ["", "--ms"]:
        out = f"/kaggle/working/released/{mode}{'_ms' if ms else ''}"
        !python eval/run_bsds_eval.py --ckpt {CKPT} --mode {mode} {ms} --store /kaggle/working/data_store/bsds  --split test --raw-root {RAW_BSDS} --out-dir {out} --nproc 4

In [ ]:
import glob, json
for f in sorted(glob.glob("/kaggle/working/released/*/scores_*.json")):
    if f.endswith("_per_image.json"): continue
    r = json.load(open(f)); print(f"{f.split('/')[-2]:12s} ODS {r['ODS']:.4f}  OIS {r['OIS']:.4f}  AP {r['AP']:.4f}")
print("published: .821/.840 (BSDS+VOC)  .813/.834 (BSDS only)")

In [ ]:
# Numeric BatchNorm check on the released weights (step 9, part 1)
!python eval/bn_check.py --numeric --ckpt third_party/PEdger/checkpoint_PEdger.pth --device cpu